# Comprobar el entorno del taller

Si es la primera vez, sigue antes el paso **0** para crear el entorno. Luego pulsa **Run All**: si todas las celdas terminan en ✅, el entorno está listo.

**Antes de empezar:** pulsa **Select Kernel** (arriba a la derecha) → **Python Environments…** → `.venv`.

## 0. Crear el virtualenv e instalar las dependencias

Solo hay que hacerlo una vez. Abre la carpeta `taller-claude` en VS Code y, en la terminal integrada (`` Ctrl+` ``):

```bash
python3 -m venv .venv
source .venv/bin/activate            # Windows: .venv\Scripts\activate
pip install -r requirements.txt
cp .env.example .env                 # y pon tu API key en .env
```

Después vuelve a este notebook, pulsa **Select Kernel** → **Python Environments…** → `.venv` y ejecuta **Run All**.

> Si `.venv` no aparece en la lista, recarga VS Code (**Developer: Reload Window**) o elige **Enter interpreter path…** y apunta a `.venv/bin/python` (Windows: `.venv\Scripts\python.exe`).

## 1. Kernel y Python

In [ ]:
import sys

print("Python:", sys.version.split()[0])
print("Ejecutable:", sys.executable)
assert sys.version_info >= (3, 10), "Hace falta Python 3.10 o superior"
assert ".venv" in sys.executable, (
    "El kernel no es el del entorno del taller: "
    "Select Kernel → Python Environments → .venv"
)
print("✅ Kernel correcto")

## 2. Librerías

In [ ]:
from importlib.metadata import version

paquetes = [
    "anthropic", "claude-agent-sdk", "ipykernel",
    "python-dotenv", "pydantic", "pandas",
]
for paquete in paquetes:
    print(f"{paquete:18} {version(paquete)}")
print("✅ Librerías instaladas")

## 3. Configuración y API key

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.append("..")   # para importar config.py desde la raíz del taller
from config import MODEL

key = os.environ.get("ANTHROPIC_API_KEY", "")
assert key.startswith("sk-ant-") and key != "sk-ant-...", (
    "Falta la API key: copia .env.example como .env y pon tu clave"
)
print("API key:", key[:10] + "…" + key[-4:])
print("Modelo:", MODEL)
print("✅ Configuración cargada")

## 4. Llamada a la API de Claude

In [ ]:
import anthropic

client = anthropic.Anthropic()
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": "Responde solo: 'Entorno listo'"}],
)
print(next(b.text for b in response.content if b.type == "text"))
print(response.usage)
print("✅ API de Claude accesible")

## 5. Claude Agent SDK dentro de Jupyter

El SDK es asíncrono: en Jupyter se usa `async for` directamente en la celda (no `asyncio.run`).

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage

options = ClaudeAgentOptions(model=MODEL, max_turns=1, setting_sources=[])

prompt = "Responde solo: 'Agent SDK listo'"
async for msg in query(prompt=prompt, options=options):
    if isinstance(msg, ResultMessage):
        print(msg.result)
        print(f"[{msg.subtype}] coste=${msg.total_cost_usd:.4f}")
        assert msg.subtype == "success"
print("✅ Agent SDK funcionando en Jupyter")

## 6. Node.js (solo para el MCP externo del bloque 4)

In [ ]:
import shutil, subprocess

npx = shutil.which("npx")
if npx:
    node = subprocess.run(
        ["node", "--version"], capture_output=True, text=True
    )
    print("Node:", node.stdout.strip())
    print("✅ Node.js disponible")
else:
    print("⚠️  No se encuentra npx: instala Node.js antes del bloque 4")